# Profiling in PyTorch (Part 1) — `matmul + add`

A demo of **roclens Cell Profile**, based on the Hugging Face post [Profiling in PyTorch (Part 1)](https://huggingface.co/blog/torch-profiler).

## How to use this notebook

- Run a cell with **Shift+Enter**. Cells using `%%rocprofv3` render a profile table **inline**.
- Or select a code cell and click the floating **Cell Profile** button to profile the whole cell.
- Then open the inline **Timeline / diagnostics** link (or **ROCm GPU Monitor → Cell Profile**) and explore, without leaving JupyterLab:
  - **Findings** — automatic diagnosis (overhead-bound vs compute-bound, idle GPU, hidden memcpy, GEMM occupancy query).
  - **Timeline** — CPU and GPU lanes (scroll to zoom, drag to pan).
  - **Dispatch chain** — the nested `aten::*` call tree.
  - **Decoded** kernel names (dtype / tile / layout).

> Run the cells top to bottom.

## 0. Setup & GPU check

In [ ]:
%load_ext roclens

In [ ]:
import torch

print(f"torch.cuda.is_available() = {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"device = {torch.cuda.get_device_name(0)}")
else:
    raise RuntimeError("No ROCm GPU visible to PyTorch — fix the runtime before profiling.")

## 1. Small `matmul + add` — overhead-bound

Tiny 64×64 work launched many times: the CPU spends more time *launching* kernels than the GPU spends *computing*. Expect the **Findings** to report **overhead-bound** and **GPU mostly idle**.

In [ ]:
%%rocprofv3 --label "matmul+add 64 (eager)" --shapes
import torch

def fn(x, w, b):
    return torch.add(torch.matmul(x, w), b)

N = 64
x = torch.randn(N, N, device="cuda")
w = torch.randn(N, N, device="cuda")
b = torch.randn(N, N, device="cuda")

# Many tiny ops the CPU has to launch one by one.
for _ in range(50):
    y = fn(x, w, b)
torch.cuda.synchronize()
print("small matmul+add done", y.shape)

## 2. Large `matmul + add` — compute-bound

4096×4096: now the GPU kernel dominates. Expect a **compute-bound** finding and one fat GEMM in the GPU lane of the **Timeline**.

In [ ]:
%%rocprofv3 --label "matmul+add 4096 (eager)"
import torch

N = 4096
x = torch.randn(N, N, device="cuda")
w = torch.randn(N, N, device="cuda")
b = torch.randn(N, N, device="cuda")

for _ in range(3):  # warm up + a few iterations
    y = torch.add(torch.matmul(x, w), b)
torch.cuda.synchronize()
print("large matmul+add done", y.shape)

## 3. `torch.compile` the same op

Inductor rewrites `add(matmul(...))` into a single **`aten::addmm`** dispatch (bias folded into the GEMM epilogue). Compare the **Dispatch chain** and **GPU kernels** with cell 2 — the GEMM kernel name is usually identical, and you may spot a device-to-device **memcpy** (the bias copy) in the **Findings**. `--trace` keeps the chrome trace so you can still **Download trace** for Perfetto if you want.

In [ ]:
%%rocprofv3 --label "matmul+add 4096 (compiled)" --trace
import torch

N = 4096
x = torch.randn(N, N, device="cuda")
w = torch.randn(N, N, device="cuda")
b = torch.randn(N, N, device="cuda")

fn_c = torch.compile(lambda x, w, b: torch.add(torch.matmul(x, w), b))
for _ in range(5):  # first iter pays the compile cost
    y = fn_c(x, w, b)
torch.cuda.synchronize()
print("compiled matmul+add done", y.shape)

## What to look for

- Cells 1 vs 2: the **Findings** flip from *overhead-bound* to *compute-bound*.
- Open the **Timeline** and hover kernels; open **Dispatch chain** to see `aten::matmul → aten::mm`.
- Cell 3: the bias `add` disappears into `aten::addmm`.

Next: **`torch-mlp-fusion-part2-mlp.ipynb`**.